# parse an existing feed snapshot
set `snapshot_dir` to the directory produced by `01_download_snapshots`.

In [0]:
import hashlib
import json
import os
import re

from datetime import datetime
from functools import reduce

from delta.tables import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

In [0]:
%run ./utils/functions

In [0]:
%run ./utils/validations

In [0]:
dbutils.widgets.text("snapshot_dir", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "pb_threat_intelligence")

In [0]:
snapshot_dir = dbutils.widgets.get("snapshot_dir").strip().rstrip("/")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

table_name = f"{catalog}.{schema}.feed_observations"
audit_table = f"{catalog}.{schema}.feed_count_audit"

if not snapshot_dir:
    raise ValueError("Provide snapshot_dir: the run directory from 01_download_snapshots")

In [0]:
# source configuration
feed_config = {
    "feed_a": ("ads_and_trackers", "hosts"),
    "feed_b": ("malware", "adblock"),
    "feed_c": ("ads_and_trackers", "adblock"),
    "feed_d": ("ads_and_trackers", "hosts"),
}

In [0]:
# validate all four files before parsing or writing any observations.
feed_metadata = {}

for name in feed_config:
    metadata_path = os.path.join(snapshot_dir, f"{name}.metadata.json")
    with open(metadata_path, encoding="utf-8") as file:
        metadata = json.load(file)

    if metadata["download_status"] != "downloaded":
        raise RuntimeError(f"Download failed for {name}: {metadata.get('error')}")

    # Resolve relative to the selected snapshot directory for portable replay.
    path = os.path.join(snapshot_dir, f"{name}.txt")
    with open(path, "rb") as file:
        content = file.read()

    if hashlib.sha256(content).hexdigest() != metadata["sha256"]:
        raise RuntimeError(f"Snapshot SHA-256 mismatch for {name}")
    if len(content) != metadata["size_bytes"]:
        raise RuntimeError(f"Snapshot size mismatch for {name}")

    metadata["resolved_snapshot_path"] = path
    feed_metadata[name] = metadata

run_ids = {metadata["run_id"] for metadata in feed_metadata.values()}
if len(run_ids) != 1:
    raise ValueError("All four snapshots must belong to the same run_id")

In [0]:
feed_dfs = {}

for name, (category, file_format) in feed_config.items():
    metadata = feed_metadata[name]
    path = metadata["resolved_snapshot_path"]
    ingestion_ts = datetime.fromisoformat(metadata["download_finished_at"])
    run_id = metadata["run_id"]

    raw_df = (
        spark.read.text(path)
        .withColumnRenamed("value", "raw_line")
        .withColumn("feed_name", F.lit(name))
        .withColumn("ingestion_ts", F.lit(ingestion_ts))
    )

    parsed_df = parse_domains(raw_df, file_format)

    feed_dfs[name] = (
        parsed_df
        .withColumn("run_id", F.lit(run_id))
        .withColumn("feed_name", F.lit(name))
        .withColumn("category", F.lit(category))
        .withColumn("ingestion_ts", F.lit(ingestion_ts))
        .withColumn("snapshot_path", F.lit(path))
        .withColumn("snapshot_sha256", F.lit(metadata["sha256"]))
        .select(
            "run_id", "feed_name", "category", "raw_line", "domain",
            "validation_status", "ingestion_ts", "snapshot_path",
            "snapshot_sha256",
        )
    )

observations_df = reduce(
    lambda left, right: left.unionByName(right), feed_dfs.values()
)

In [0]:
validate_parsed_count(
    parsed_df=observations_df,
    run_id=run_id,
    run_date=ingestion_ts.date(),
    audit_table=audit_table,
    max_drop_ratio=0.7
)

Feed validation passed. Parsed feeds contain following counts:


{'feed_a': 72525, 'feed_b': 18105, 'feed_c': 89822, 'feed_d': 1707}

In [0]:
# Only this run is replaced; previous runs remain available.
# no deduplication or filtering of validation statuses is performed here.
(
    observations_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "false")
    .option("replaceWhere", f"run_id = '{run_id}'")
    .saveAsTable(table_name)
)
print(f"Saved run {run_id} to {table_name}")

Saved run 203832Z_0ccefab2 to workspace.pb_threat_intelligence.feed_observations


In [0]:
DeltaTable.forName(spark, audit_table).update(
    condition=(
        (F.col("run_id") == F.lit(run_id))
        & (F.col("stage") == F.lit("parsed"))
        & (F.col("status") == F.lit("logged"))
    ),
    set={
        "status": F.lit("passed")
    }
)
print(f"Updated audit table {audit_table} for run {run_id}")

Updated audit table workspace.pb_threat_intelligence.feed_count_audit for run 203832Z_0ccefab2
